Label each grid's land type

In [1]:
import rasterio
import numpy as np
from pathlib import Path

# === 文件路径 ===
lc_path = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/CLCD_v01_2012_albert_jiangsu_albers.tif")

# === 打开栅格文件 ===
with rasterio.open(lc_path) as src:
    arr = src.read(1)  # 读取第1波段
    nodata = src.nodata
    print(f"✅ CRS: {src.crs}")
    print(f"✅ Shape: {arr.shape}")
    print(f"✅ Nodata value: {nodata}")

# === 统计唯一值 ===
# 去除 nodata
valid = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.floating) else arr[arr != nodata]

unique_values, counts = np.unique(valid, return_counts=True)

# === 打印前若干个类别 ===
print("\n📊 Land Cover Categories (Value: Pixel Count):")
for val, cnt in zip(unique_values, counts):
    print(f"  {int(val):>3d} : {cnt:,}")

print(f"\nTotal unique land cover classes: {len(unique_values)}")


✅ CRS: EPSG:4547
✅ Shape: (17414, 20242)
✅ Nodata value: 0.0

📊 Land Cover Categories (Value: Pixel Count):
    1 : 80,759,008
    2 : 2,024,953
    4 : 23,659
    5 : 13,894,252
    7 : 2,605
    8 : 17,115,469

Total unique land cover classes: 6


In [4]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from pathlib import Path

# === 路径 ===
base_dir  = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
grid_path = base_dir / "Jiangsu_grid_500m_ID.gpkg"
lc_path   = base_dir / "CLCD_v01_2012_albert_jiangsu_albers.tif"
out_xlsx  = base_dir / "Jiangsu_grid_landcover_share_2012.xlsx"

# === Land cover class names 对照表 ===
lc_classes = {
    1: "Cropland",
    2: "Forest",
    3: "Shrub",
    4: "Grassland",
    5: "Water",
    6: "SnowIce",
    7: "Barren",
    8: "Impervious",
    9: "Wetland"
}

# === 1. 读取网格 ===
grid = gpd.read_file(grid_path)
print(f"✅ Loaded grid: {len(grid)} cells")

# === 2. zonal_stats，按类别统计像元数 ===
print("🔹 Computing zonal statistics ...")
stats = zonal_stats(
    vectors=grid,
    raster=str(lc_path),
    categorical=True,
    nodata=255,
    geojson_out=False
)

# === 3. 将结果整理成 DataFrame ===
records = []
for s in stats:
    # 总像元数（排除 nodata）
    total = sum(s.get(k, 0) for k in lc_classes.keys())

    # 若格子完全为空
    if total == 0:
        row = {lc_classes[k]: 0.0 for k in lc_classes}
    else:
        row = {lc_classes[k]: s.get(k, 0) / total for k in lc_classes}

    records.append(row)

share_df = pd.DataFrame(records)
share_df.insert(0, "grid_id", grid["grid_id"])

print("📄 Example rows:")
print(share_df.head())

# === 4. 保存 Excel ===
share_df.to_excel(out_xlsx, index=False)
print(f"💾 Saved land-cover share table to:\n{out_xlsx}")


✅ Loaded grid: 410620 cells
🔹 Computing zonal statistics ...
📄 Example rows:
   grid_id  Cropland  Forest  Shrub  Grassland  Water  SnowIce  Barren  \
0    78274  0.852217     0.0    0.0        0.0    0.0      0.0     0.0   
1    78275  0.775610     0.0    0.0        0.0    0.0      0.0     0.0   
2    78278  1.000000     0.0    0.0        0.0    0.0      0.0     0.0   
3    78279  0.926471     0.0    0.0        0.0    0.0      0.0     0.0   
4    78280  0.832031     0.0    0.0        0.0    0.0      0.0     0.0   

   Impervious  Wetland  
0    0.147783      0.0  
1    0.224390      0.0  
2    0.000000      0.0  
3    0.073529      0.0  
4    0.167969      0.0  
💾 Saved land-cover share table to:
/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_landcover_share_2012.xlsx
